<a href="https://colab.research.google.com/github/saniluttu/used-car-data-preprocessing-day12/blob/main/Used_Car_Preprocessing_Day12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Used Car Data Preprocessing
**Day 12 Assignment**

This notebook performs a complete data preprocessing workflow on the Used Car Resale dataset: identifying and handling outliers using the IQR method, encoding categorical variables with nominal and ordinal techniques, applying feature scaling, splitting the data into training and testing sets, and fitting all transformations only on the training data to avoid data leakage.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

pd.set_option('display.max_columns', None)
print("Pandas version:", pd.__version__)

Pandas version: 3.0.2


## 1. Loading and Understanding the Dataset

In [ ]:
df = pd.read_csv('Day12_Used_Car_Preprocessing_Dataset.csv')
print("Shape:", df.shape)
df.head()

Shape: (320, 15)


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    str    
 1   Brand               320 non-null    str    
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    str    
 7   Transmission        320 non-null    str    
 8   City                320 non-null    str    
 9   Seller_Type         320 non-null    str    
 10  Condition           320 non-null    str    
 11  Previous_Owners     320 non-null    int64  
 12  Accidents_Reported  320 non-null    int64  
 13  Service_Score       320 non-null    int64  
 14  Resale_Price_Lakh   320 non-null    float64
dtypes: float64(2), int64(6), str(7)
memory usage: 37.6 KB


In [ ]:
df.isna().sum()

Car_ID                0
Brand                 0
Year                  0
Mileage_Km            0
Engine_CC             0
Power_BHP             0
Fuel_Type             0
Transmission          0
City                  0
Seller_Type           0
Condition             0
Previous_Owners       0
Accidents_Reported    0
Service_Score         0
Resale_Price_Lakh     0
dtype: int64

In [ ]:
df.describe()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
count,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000
mean,2019.537500,74110.203125,1346.703125,150.489688,1.668750,0.243750,76.203125,4.963031
std,3.341367,38885.260771,543.408160,36.665353,0.865369,0.528164,12.745864,3.359259
min,2014.000000,700.000000,600.000000,51.400000,1.000000,0.000000,55.000000,1.200000
25%,2017.000000,46323.250000,1004.750000,128.450000,1.000000,0.000000,64.750000,2.277500
50%,2020.000000,72718.500000,1303.000000,150.750000,1.000000,0.000000,77.000000,4.610000
75%,2022.000000,97951.500000,1635.250000,171.475000,2.000000,0.000000,87.000000,6.835000
max,2025.000000,320000.000000,5000.000000,390.000000,4.000000,2.000000,98.000000,28.500000


## 2. Identifying Outliers (IQR Method)
Using the Interquartile Range (IQR) method to detect outliers in key numerical columns. A value is flagged as an outlier if it falls below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR`.

In [ ]:
numerical_cols = ['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP',
                   'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Resale_Price_Lakh']

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    print(f"{col}: {len(outliers)} outliers | bounds = [{lower:.2f}, {upper:.2f}]")

Year: 0 outliers | bounds = [2009.50, 2029.50]
Mileage_Km: 2 outliers | bounds = [-31119.12, 175393.88]
Engine_CC: 6 outliers | bounds = [59.00, 2581.00]
Power_BHP: 7 outliers | bounds = [63.91, 236.01]
Previous_Owners: 14 outliers | bounds = [-0.50, 3.50]
Accidents_Reported: 63 outliers | bounds = [0.00, 0.00]
Service_Score: 0 outliers | bounds = [31.38, 120.38]
Resale_Price_Lakh: 5 outliers | bounds = [-4.56, 13.67]


In [ ]:
# Closer look at the columns with the most outliers
mileage_outliers, mileage_lower, mileage_upper = detect_outliers_iqr(df, 'Mileage_Km')
price_outliers, price_lower, price_upper = detect_outliers_iqr(df, 'Resale_Price_Lakh')

print("Mileage_Km outliers:")
print(mileage_outliers[['Car_ID', 'Mileage_Km']])
print("\nResale_Price_Lakh outliers:")
print(price_outliers[['Car_ID', 'Resale_Price_Lakh']])

Mileage_Km outliers:
     Car_ID  Mileage_Km
31  CAR0032      320000
67  CAR0068      280000

Resale_Price_Lakh outliers:
      Car_ID  Resale_Price_Lakh
31   CAR0032               15.0
149  CAR0150               22.5
188  CAR0189               18.7
227  CAR0228               28.5
261  CAR0262               16.9


## 3. Handling Outliers
`Mileage_Km` and `Resale_Price_Lakh` show the most extreme values. Rather than dropping these rows outright (which would lose real, valid high-end cars and high-mileage cars), we **cap (winsorize)** them at the IQR bounds — this reduces the influence of extreme values while preserving all records for modeling.

In [ ]:
df_clean = df.copy()

def cap_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = data[column].clip(lower=lower_bound, upper=upper_bound)
    return data

for col in ['Mileage_Km', 'Resale_Price_Lakh', 'Engine_CC', 'Power_BHP']:
    df_clean = cap_outliers_iqr(df_clean, col)

print("Outliers capped. Updated summary:")
df_clean[['Mileage_Km', 'Resale_Price_Lakh', 'Engine_CC', 'Power_BHP']].describe()

Outliers capped. Updated summary:


,Mileage_Km,Resale_Price_Lakh,Engine_CC,Power_BHP
count,320.000000,320.000000,320.000000,320.000000
mean,73331.414844,4.859145,1323.546875,149.726211
std,35402.678568,2.899592,443.995069,32.576563
min,700.000000,1.200000,600.000000,63.912500
25%,46323.250000,2.277500,1004.750000,128.450000
50%,72718.500000,4.610000,1303.000000,150.750000
75%,97951.500000,6.835000,1635.250000,171.475000
max,175393.875000,13.671250,2581.000000,236.012500


In [ ]:
# Confirm no more outliers remain by the same IQR rule
for col in ['Mileage_Km', 'Resale_Price_Lakh', 'Engine_CC', 'Power_BHP']:
    outliers, _, _ = detect_outliers_iqr(df_clean, col)
    print(f"{col}: {len(outliers)} outliers remaining after capping")

Mileage_Km: 0 outliers remaining after capping
Resale_Price_Lakh: 0 outliers remaining after capping
Engine_CC: 0 outliers remaining after capping
Power_BHP: 0 outliers remaining after capping


## 4. Separating Features and Target Variable
`Resale_Price_Lakh` is the target variable we want to predict. `Car_ID` is dropped since it's just a unique identifier with no predictive value.

In [ ]:
X = df_clean.drop(columns=['Car_ID', 'Resale_Price_Lakh'])
y = df_clean['Resale_Price_Lakh']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
X.head()

Features shape: (320, 13)
Target shape: (320,)


,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score
0,Skoda,2021,69708.0,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72
1,Toyota,2020,88881.0,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87
2,Volkswagen,2021,43646.0,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90
3,Tata,2019,70847.0,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66
4,Tata,2016,101228.0,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84


## 5. Splitting the Data (Before Encoding/Scaling)
The train-test split is done **before** fitting any encoders or scalers, so that all transformations are learned only from the training data — preventing information from the test set leaking into preprocessing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Training set shape: (256, 13)
Testing set shape: (64, 13)


## 6. Encoding Categorical Variables

- **`Condition`** is an ordinal variable (Poor < Fair < Good < Very Good < Excellent) — encoded with `OrdinalEncoder` respecting that natural order.
- **`Brand`, `Fuel_Type`, `Transmission`, `City`, `Seller_Type`** are nominal (no inherent order) — encoded with one-hot encoding (`pd.get_dummies`).

All encoders are fitted only on `X_train`, then applied to both `X_train` and `X_test`.

In [ ]:
condition_order = [['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']]

ordinal_encoder = OrdinalEncoder(categories=condition_order)
X_train_cond = X_train.copy()
X_test_cond = X_test.copy()

# Fit only on training data
ordinal_encoder.fit(X_train[['Condition']])

X_train_cond['Condition'] = ordinal_encoder.transform(X_train[['Condition']])
X_test_cond['Condition'] = ordinal_encoder.transform(X_test[['Condition']])

X_train_cond[['Condition']].head()

,Condition
132,3.0
317,2.0
234,2.0
312,2.0
232,3.0


In [ ]:
nominal_cols = ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']

# Determine categories from training data only, then apply the same columns to both sets
X_train_encoded = pd.get_dummies(X_train_cond, columns=nominal_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_cond, columns=nominal_cols, drop_first=True)

# Align test set columns to match training set (handles any category missing in one split)
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("Training features shape after encoding:", X_train_encoded.shape)
print("Testing features shape after encoding:", X_test_encoded.shape)
X_train_encoded.head()

Training features shape after encoding: (256, 32)
Testing features shape after encoding: (64, 32)


,Year,Mileage_Km,Engine_CC,Power_BHP,Condition,Previous_Owners,Accidents_Reported,Service_Score,Brand_Hyundai,Brand_Kia,Brand_Mahindra,Brand_Maruti,Brand_Renault,Brand_Skoda,Brand_Tata,Brand_Toyota,Brand_Volkswagen,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Manual,City_Bengaluru,City_Chandigarh,City_Delhi,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Dealer,Seller_Type_Individual
132,2018,65330.0,1188,159.8,3.0,1,0,71,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False
317,2022,76484.0,1038,139.6,2.0,1,1,69,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False
234,2015,112800.0,1279,163.5,2.0,1,0,70,False,False,False,False,True,False,False,False,False,False,False,True,True,False,False,False,False,False,True,False,False,False,False,True
312,2023,66420.0,1549,163.3,2.0,2,0,79,False,False,True,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,True,False
232,2016,98325.0,1687,136.2,3.0,2,0,78,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True


## 7. Feature Scaling
Numerical features are standardized using `StandardScaler` (zero mean, unit variance). The scaler is fitted **only on the training data**, then used to transform both sets — keeping the test set genuinely unseen during preprocessing.

In [ ]:
numerical_features = ['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP',
                       'Previous_Owners', 'Accidents_Reported', 'Service_Score']

scaler = StandardScaler()

# Fit only on training data
scaler.fit(X_train_encoded[numerical_features])

X_train_final = X_train_encoded.copy()
X_test_final = X_test_encoded.copy()

X_train_final[numerical_features] = scaler.transform(X_train_encoded[numerical_features])
X_test_final[numerical_features] = scaler.transform(X_test_encoded[numerical_features])

X_train_final[numerical_features].describe().round(2)

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score
count,256.00,256.00,256.00,256.00,256.00,256.00,256.00
mean,-0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.70,-2.05,-1.63,-2.70,-0.73,-0.44,-1.74
25%,-0.79,-0.77,-0.71,-0.64,-0.73,-0.44,-0.88
50%,0.12,-0.01,-0.06,0.01,-0.73,-0.44,0.11
75%,0.73,0.69,0.69,0.66,0.43,-0.44,0.75
max,1.63,2.88,2.78,2.69,2.77,3.40,1.71


## 8. Verifying the Processed Dataset

In [ ]:
print("Final training set shape:", X_train_final.shape)
print("Final testing set shape:", X_test_final.shape)
print("\nAny missing values in training set:", X_train_final.isna().sum().sum())
print("Any missing values in testing set:", X_test_final.isna().sum().sum())
print("\nData types after preprocessing:")
print(X_train_final.dtypes.value_counts())

Final training set shape: (256, 32)
Final testing set shape: (64, 32)

Any missing values in training set: 0
Any missing values in testing set: 0

Data types after preprocessing:
bool       24
float64     8
Name: count, dtype: int64


In [ ]:
X_train_final.head()

,Year,Mileage_Km,Engine_CC,Power_BHP,Condition,Previous_Owners,Accidents_Reported,Service_Score,Brand_Hyundai,Brand_Kia,Brand_Mahindra,Brand_Maruti,Brand_Renault,Brand_Skoda,Brand_Tata,Brand_Toyota,Brand_Volkswagen,Fuel_Type_Diesel,Fuel_Type_Electric,Fuel_Type_Petrol,Transmission_Manual,City_Bengaluru,City_Chandigarh,City_Delhi,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Dealer,Seller_Type_Individual
132,-0.486391,-0.225904,-0.322825,0.303144,3.0,-0.734379,-0.442634,-0.454105,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False,True,False,False,False,False
317,0.725445,0.089296,-0.656909,-0.328921,2.0,-0.734379,1.477948,-0.614672,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False
234,-1.395268,1.115547,-0.120148,0.418918,2.0,-0.734379,-0.442634,-0.534389,False,False,False,False,True,False,False,False,False,False,False,True,True,False,False,False,False,False,True,False,False,False,False,True
312,1.028404,-0.195102,0.481202,0.412660,2.0,0.433329,-0.442634,0.188165,False,False,True,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,True,False
232,-1.092309,0.706499,0.788558,-0.435309,3.0,0.433329,-0.442634,0.107881,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,True


In [ ]:
# Combine features and target back together for saving
train_processed = X_train_final.copy()
train_processed['Resale_Price_Lakh'] = y_train.values

test_processed = X_test_final.copy()
test_processed['Resale_Price_Lakh'] = y_test.values

print("Processed training set shape:", train_processed.shape)
print("Processed testing set shape:", test_processed.shape)

Processed training set shape: (256, 33)
Processed testing set shape: (64, 33)


## 9. Exporting the Preprocessed Dataset

In [ ]:
train_processed.to_csv('Used_Car_Train_Processed.csv', index=False)
test_processed.to_csv('Used_Car_Test_Processed.csv', index=False)
print("Exported Used_Car_Train_Processed.csv and Used_Car_Test_Processed.csv")

Exported Used_Car_Train_Processed.csv and Used_Car_Test_Processed.csv


## Summary of Preprocessing Decisions

1. **Outliers** in `Mileage_Km`, `Resale_Price_Lakh`, `Engine_CC`, and `Power_BHP` were identified using the IQR method and **capped** (not removed) at the IQR bounds, preserving all 320 records while reducing the influence of extreme values.
2. **`Condition`** (ordinal: Poor → Excellent) was encoded with `OrdinalEncoder`, respecting its natural order.
3. **`Brand`, `Fuel_Type`, `Transmission`, `City`, `Seller_Type`** (nominal, no inherent order) were one-hot encoded.
4. **Numerical features** were standardized with `StandardScaler` (zero mean, unit variance).
5. The dataset was **split into training (80%) and testing (20%) sets before** any encoding or scaling was fitted, and all transformations were **fitted only on the training data**, then applied to the test set — avoiding data leakage.
6. The final processed training and testing sets were verified for shape, missing values, and data types, then exported as separate CSV files.